In [1]:
!pip install -q torch \
 transformers \
 bitsandbytes \
 nvidia-ml-py \
 langchain-groq \
 langchain-google-genai \
 langchain-mistralai

In [2]:
!pip install -q --upgrade transformers accelerate

In [6]:
import torch
import time
import re
import threading
import pandas as pd
import pynvml
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_mistralai import ChatMistralAI
from langchain.messages import HumanMessage

print("Imports successful")

Imports successful


In [7]:
! nvidia-smi

Sat Jun 27 09:00:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
df = pd.read_json('benchmark.json')
print(df["prompt"][1])

If a clock strikes 6 times in 5 seconds, how many seconds will it take to strike 12 times?


In [10]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",                      # Use NF4 data type for higher precision
    bnb_4bit_use_double_quant=True,                 # Nested quantization to save extra VRAM
    bnb_4bit_compute_dtype=torch.bfloat16           # Speeds up processing if GPU supports it
)

In [11]:
tokenizer_deepseek = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")
model_deepseek = AutoModelForCausalLM.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
                                             dtype=torch.float16,
                                            quantization_config = quantization_config,
                                             device_map ='auto')

config.json:   0%|          | 0.00/679 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.07k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [9]:

# ── Quantization config ──────────────────────────────────────────────
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

# ── Load model ───────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")
model = AutoModelForCausalLM.from_pretrained(
    "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    quantization_config=quantization_config,
    device_map='auto'
)

# ── GPU monitor setup ────────────────────────────────────────────────
pynvml.nvmlInit()
handle = pynvml.nvmlDeviceGetHandleByIndex(0)

# ── Helper: poll GPU in background thread during generation ──────────
def poll_gpu(handle, interval, results, stop_event):
    """Polls GPU every `interval` seconds until stop_event is set."""
    util_readings, temp_readings, mem_readings = [], [], []
    while not stop_event.is_set():
        util_readings.append(pynvml.nvmlDeviceGetUtilizationRates(handle).gpu)
        temp_readings.append(pynvml.nvmlDeviceGetTemperature(handle, pynvml.NVML_TEMPERATURE_GPU))
        mem_readings.append(pynvml.nvmlDeviceGetMemoryInfo(handle).used / 1024**2)
        time.sleep(interval)
    results['gpu_usage_pct']   = round(max(util_readings), 2)   # peak usage
    results['gpu_temp_c']      = round(max(temp_readings), 2)   # peak temp
    results['gpu_mem_used_mb'] = round(max(mem_readings), 2)    # peak memory

# ── Strip DeepSeek <think> chain-of-thought block ───────────────────
def strip_think_block(text):
    return re.sub(r'<think>.*?</think>', '', str(text), flags=re.DOTALL).strip()

# ── Main inference loop ──────────────────────────────────────────────
deepseek_records = []

for i in range(len(df)):
    inputs = tokenizer(
        df["prompt"][i],
        return_tensors='pt'
    ).to(model.device)

    prompt_tokens = inputs["input_ids"].shape[1]

    # Start background GPU polling
    gpu_results = {}
    stop_event  = threading.Event()
    gpu_thread  = threading.Thread(
        target=poll_gpu,
        args=(handle, 0.5, gpu_results, stop_event)  # poll every 0.5s
    )
    gpu_thread.start()

    # Generation
    start = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=500,
            pad_token_id=tokenizer.eos_token_id   # avoids warning on Qwen
        )
    latency = time.time() - start

    # Stop GPU polling
    stop_event.set()
    gpu_thread.join()

    # Decode only new tokens (strips prompt echo)
    new_tokens    = outputs[0][prompt_tokens:]
    output_tokens = new_tokens.shape[0]
    text          = tokenizer.decode(new_tokens, skip_special_tokens=True)
    text          = strip_think_block(text)

    deepseek_records.append({
        "question_id"     : i,
        "question"        : df["prompt"][i],
        "answer"          : text,
        "prompt_tokens"   : prompt_tokens,
        "output_tokens"   : output_tokens,
        "total_tokens"    : prompt_tokens + output_tokens,
        "latency_sec"     : round(latency, 4),
        "tokens_per_sec"  : round(output_tokens / latency, 2),
        "gpu_usage_pct"   : gpu_results.get('gpu_usage_pct'),
        "gpu_temp_c"      : gpu_results.get('gpu_temp_c'),
        "gpu_mem_used_mb" : gpu_results.get('gpu_mem_used_mb'),
    })

    print(f"[{i+1}/{len(df)}] done | "
          f"out_tokens: {output_tokens} | "
          f"latency: {latency:.2f}s | "
          f"tok/s: {output_tokens/latency:.1f} | "
          f"gpu: {gpu_results.get('gpu_usage_pct')}% | "
          f"temp: {gpu_results.get('gpu_temp_c')}C | "
          f"mem: {gpu_results.get('gpu_mem_used_mb')}MB")

deepseek_df = pd.DataFrame(deepseek_records)
deepseek_df['model'] = 'deepseek'
deepseek_df.to_csv('deepseek_metrics.csv', index=False)
print("\nSaved deepseek_metrics.csv")
print(deepseek_df[['question_id','output_tokens','latency_sec',
                    'tokens_per_sec','gpu_usage_pct','gpu_temp_c',
                    'gpu_mem_used_mb']].to_string())

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[1/30] done | out_tokens: 500 | latency: 39.68s | tok/s: 12.6 | gpu: 47% | temp: 43C | mem: 2218.19MB
[2/30] done | out_tokens: 500 | latency: 66.83s | tok/s: 7.5 | gpu: 49% | temp: 46C | mem: 2218.19MB
[3/30] done | out_tokens: 500 | latency: 38.33s | tok/s: 13.0 | gpu: 46% | temp: 48C | mem: 2218.19MB
[4/30] done | out_tokens: 500 | latency: 41.09s | tok/s: 12.2 | gpu: 49% | temp: 49C | mem: 2218.19MB
[5/30] done | out_tokens: 500 | latency: 38.10s | tok/s: 13.1 | gpu: 46% | temp: 50C | mem: 2218.19MB
[6/30] done | out_tokens: 147 | latency: 11.41s | tok/s: 12.9 | gpu: 41% | temp: 51C | mem: 2218.19MB
[7/30] done | out_tokens: 500 | latency: 42.22s | tok/s: 11.8 | gpu: 46% | temp: 51C | mem: 2218.19MB
[8/30] done | out_tokens: 262 | latency: 20.37s | tok/s: 12.9 | gpu: 43% | temp: 52C | mem: 2218.19MB
[9/30] done | out_tokens: 500 | latency: 38.25s | tok/s: 13.1 | gpu: 46% | temp: 52C | mem: 2218.19MB
[10/30] done | out_tokens: 500 | latency: 37.79s | tok/s: 13.2 | gpu: 55% | temp: 5

In [10]:
llm_groq = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.4,
    api_key=userdata.get("GROQ_API_KEY")
)

groq_records = []

for i in range(len(df)):
    start = time.time()
    output = llm_groq.invoke(df["prompt"][i])
    latency = time.time() - start

    # Groq gives token usage directly in metadata
    usage = output.response_metadata.get('token_usage', {})
    prompt_tokens    = usage.get('prompt_tokens', 0)
    output_tokens    = usage.get('completion_tokens', 0)
    total_tokens     = usage.get('total_tokens', 0)
    # use groq's own time if available, else use ours
    groq_latency     = usage.get('completion_time', latency)

    groq_records.append({
        "question_id"      : i,
        "question"         : df["prompt"][i],
        "answer"           : output.content,
        "prompt_tokens"    : prompt_tokens,
        "output_tokens"    : output_tokens,
        "total_tokens"     : total_tokens,
        "latency_sec"      : round(groq_latency, 4),
        "tokens_per_sec"   : round(output_tokens / groq_latency, 2) if groq_latency > 0 else 0,
        "gpu_usage_pct"    : None,   # cloud API — not applicable
        "gpu_temp_c"       : None,
        "gpu_mem_used_mb"  : None,
    })

    print(f"[{i+1}/{len(df)}] done | tokens: {total_tokens} | latency: {groq_latency:.2f}s")

groq_df = pd.DataFrame(groq_records)
groq_df.to_csv('groq_metrics.csv', index=False)
print(groq_df.head())

[1/30] done | tokens: 293 | latency: 0.29s
[2/30] done | tokens: 197 | latency: 0.16s
[3/30] done | tokens: 286 | latency: 0.34s
[4/30] done | tokens: 206 | latency: 0.15s
[5/30] done | tokens: 157 | latency: 0.12s
[6/30] done | tokens: 323 | latency: 0.45s
[7/30] done | tokens: 401 | latency: 0.40s
[8/30] done | tokens: 149 | latency: 0.17s
[9/30] done | tokens: 234 | latency: 0.19s
[10/30] done | tokens: 280 | latency: 0.29s
[11/30] done | tokens: 332 | latency: 0.35s
[12/30] done | tokens: 406 | latency: 0.51s
[13/30] done | tokens: 427 | latency: 0.59s
[14/30] done | tokens: 344 | latency: 0.36s
[15/30] done | tokens: 661 | latency: 0.80s
[16/30] done | tokens: 187 | latency: 0.23s
[17/30] done | tokens: 572 | latency: 0.75s
[18/30] done | tokens: 530 | latency: 0.65s
[19/30] done | tokens: 221 | latency: 0.25s
[20/30] done | tokens: 932 | latency: 1.26s
[21/30] done | tokens: 145 | latency: 0.28s
[22/30] done | tokens: 89 | latency: 0.07s
[23/30] done | tokens: 119 | latency: 0.11

In [16]:
llm_gemini = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0.4,
    max_tokens=1024,
    api_key=userdata.get("GEMINI_API_KEY")
)

gemini_records = []

for i in range(len(df)):
    start = time.time()
    output = llm_gemini.invoke(df["prompt"][i])
    latency = time.time() - start

    # Gemini gives token usage in usage_metadata
    prompt_tokens = output.usage_metadata.get('input_tokens', 0)
    output_tokens = output.usage_metadata.get('output_tokens', 0)
    total_tokens  = output.usage_metadata.get('total_tokens', 0)

    # extract text safely
    if isinstance(output.content, list):
        answer = output.content[0].get('text', '')
    else:
        answer = output.content

    gemini_records.append({
        "question_id"      : i,
        "question"         : df["prompt"][i],
        "answer"           : answer,
        "prompt_tokens"    : prompt_tokens,
        "output_tokens"    : output_tokens,
        "total_tokens"     : total_tokens,
        "latency_sec"      : round(latency, 4),
        "tokens_per_sec"   : round(output_tokens / latency, 2) if latency > 0 else 0,
        "gpu_usage_pct"    : None,   # cloud API — not applicable
        "gpu_temp_c"       : None,
        "gpu_mem_used_mb"  : None,
    })

    print(f"[{i+1}/{len(df)}] done | tokens: {total_tokens} | latency: {latency:.2f}s")

gemini_df = pd.DataFrame(gemini_records)
gemini_df.to_csv('gemini_metrics.csv', index=False)
print(gemini_df.head())

[1/30] done | tokens: 287 | latency: 1.50s
[2/30] done | tokens: 191 | latency: 1.42s
[3/30] done | tokens: 355 | latency: 2.55s
[4/30] done | tokens: 122 | latency: 1.14s
[5/30] done | tokens: 125 | latency: 1.39s
[6/30] done | tokens: 256 | latency: 1.67s
[7/30] done | tokens: 346 | latency: 2.81s
[8/30] done | tokens: 180 | latency: 11.97s
[9/30] done | tokens: 291 | latency: 3.26s
[10/30] done | tokens: 351 | latency: 1.49s
[11/30] done | tokens: 317 | latency: 1.93s
[12/30] done | tokens: 419 | latency: 2.62s
[13/30] done | tokens: 465 | latency: 2.45s
[14/30] done | tokens: 421 | latency: 4.40s
[15/30] done | tokens: 750 | latency: 9.55s
[16/30] done | tokens: 632 | latency: 3.87s
[17/30] done | tokens: 683 | latency: 5.35s
[18/30] done | tokens: 651 | latency: 3.16s
[19/30] done | tokens: 311 | latency: 1.93s
[20/30] done | tokens: 730 | latency: 3.68s
[21/30] done | tokens: 75 | latency: 1.23s
[22/30] done | tokens: 52 | latency: 0.95s
[23/30] done | tokens: 77 | latency: 0.88s